# 🧠 Laboratorio de Deep Learning: la red que aprende sola (y el detector de dias raros)

**Diplomado en Inteligencia Artificial · UNAC · Clase 04 · Plantilla del aula**

---

### 🎯 Que vamos a hacer

- **Primero:** armar una red neuronal que mire cinco datos de un cliente y diga si se va a ir. Es el mismo problema del notebook anterior, pero ahora resuelto con la tecnologia que esta detras de ChatGPT.
- **Despues:** el plato fuerte. Un detector de **dias raros** en la caja de una tienda: le mostramos a la red 950 dias normales, y ella nos senala cuales de los 1.000 dias **no se parecen** a un dia normal.
- **Al final:** usted se lleva una lista de diez dias para auditar, ordenados de mas sospechoso a menos. Eso es lo que se lleva a la reunion del lunes.
- **La frase que se queda:** no le ensenamos a la maquina como se ve un fraude. Le ensenamos como se ve un dia normal.

### 🧭 Como se usa

**Usted no tiene que escribir una sola linea de codigo.**

1. Corra las celdas **una por una** con el boton de play ▶ que aparece a la izquierda de cada bloque gris.
2. O corra todo de una vez con **Entorno de ejecucion → Ejecutar todas**.
3. Lo **UNICO** que se cambia en este notebook esta en el **PANEL DE CONTROL** de la celda de abajo. Todo lo demas ya viene resuelto.

> Este notebook usa **TensorFlow**, que ya viene instalado en Google Colab. No hay que instalar nada, no hay que reiniciar nada y **no hace falta GPU**: todo corre en unos segundos con el computador gratis de Colab.

## 🎛️ PANEL DE CONTROL

> # ⚠️ AQUI Y SOLO AQUI SE CAMBIAN COSAS

Mueva las perillas de abajo y vuelva a correr el notebook completo. Si algo se le desordena,
ponga los valores originales (800 · 1000 · 5 · 2.0 · 100) y todo vuelve a la normalidad.

| Perilla | Que significa en la vida real |
|---|---|
| `NUM_CLIENTES` | Cuantos clientes tiene la base de la primera parte. |
| `NUM_DIAS` | Cuantos dias de operacion de la tienda vamos a revisar. |
| `PORCENTAJE_DIAS_MANIPULADOS` | Que tan sucio esta el negocio: cuantos dias de cada 100 tienen la caja manipulada. |
| `SENSIBILIDAD_DEL_DETECTOR` | Que tan desconfiado es el detector. **Bajelo** (1.0) y marca mas dias, incluidos buenos. **Subalo** (3.0) y solo marca lo escandaloso. |
| `EPOCAS_MAXIMAS` | Cuantas vueltas de estudio le damos a la red antes de mandarla a trabajar. |

In [ ]:
#@title 🎛️ PANEL DE CONTROL — corra esta celda despues de mover las perillas

NUM_CLIENTES = 800  #@param {type:"slider", min:400, max:2000, step:100}
NUM_DIAS = 1000  #@param {type:"slider", min:500, max:3000, step:100}
PORCENTAJE_DIAS_MANIPULADOS = 5  #@param {type:"slider", min:1, max:15, step:1}
SENSIBILIDAD_DEL_DETECTOR = 2.0  #@param {type:"slider", min:1.0, max:3.0, step:0.5}
EPOCAS_MAXIMAS = 100  #@param {type:"slider", min:20, max:300, step:10}

print("Panel de control listo:")
print(" - Clientes de la Parte A .........", NUM_CLIENTES)
print(" - Dias de tienda de la Parte B ...", NUM_DIAS)
print(" - Dias manipulados ...............", PORCENTAJE_DIAS_MANIPULADOS, "%")
print(" - Sensibilidad del detector ......", SENSIBILIDAD_DEL_DETECTOR, "desviaciones")
print(" - Vueltas maximas de estudio .....", EPOCAS_MAXIMAS)

## 🧰 Preparacion: traer las herramientas

Esta celda solo abre la caja de herramientas: TensorFlow (la libreria de redes neuronales de Google),
pandas para las tablas y matplotlib para las graficas. Ya vienen instaladas en Colab.

Tambien fijamos la **semilla 42**. Eso hace que los numeros "al azar" salgan iguales cada vez, para que
lo que usted ve en su pantalla sea lo mismo que se ve en el proyector del salon.

**Que mirar en el resultado:** una linea que diga `TensorFlow 2.x listo`. Si aparece, ya esta todo puesto.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # menos mensajes tecnicos de TensorFlow en pantalla

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

np.random.seed(42)          # semilla de numpy: los datos inventados salen iguales cada vez
tf.random.set_seed(42)      # semilla de TensorFlow: la red arranca siempre con los mismos pesos

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

print("TensorFlow", tf.__version__, "listo.")
print("Todo corre en CPU: no hace falta pedir GPU en Colab.")

---

# 🅰️ PARTE A · La red neuronal que clasifica

Antes de meternos con el detector de fraude, hay que ver **que es** una red neuronal trabajando.
Vamos a hacerlo con un problema que ya conocemos: **quien se va a ir de mis clientes**.

Una red neuronal es una tabla de multiplicaciones muy grande que se corrige a si misma. Le mostramos
miles de casos con la respuesta al lado, ella se equivoca, mide cuanto se equivoco y ajusta los numeros
internos. Otra vez. Y otra vez. Eso es todo el "aprendizaje profundo".

## 1️⃣ ¿Puede una red neuronal decirme quien se va a ir?

Armamos una base de clientes con cinco datos que cualquier negocio tiene a mano: cuanto compra al mes,
hace cuantos dias no viene, cuanto gasta por compra, cuantas quejas puso y hace cuanto es cliente.

La columna `se_va` es la respuesta que ya conocemos del pasado (1 = se fue, 0 = se quedo). Los datos son
inventados aqui mismo, pero **con trampa a proposito**: un 8% de los clientes rompe la regla, porque en la
vida real la gente tambien se va sin motivo aparente. Si los datos fueran perfectos, la red acertaria el
100% y usted saldria de esta clase con una idea falsa.

**Que mirar en el resultado:** cuantos clientes hay y cuantos se fueron (deberia ser cerca de un tercio),
y la tablita de abajo con los cinco primeros clientes.

In [ ]:
rng = np.random.default_rng(42)
n = NUM_CLIENTES

compras_mes      = rng.normal(6, 2.2, n).clip(0, None).round(1)          # compras que hace al mes
dias_sin_venir   = rng.gamma(2.0, 14, n).clip(0, 200).round(0)           # dias desde la ultima compra
ticket_promedio  = rng.normal(45000, 14000, n).clip(9000, None).round(0) # pesos por compra
quejas_6meses    = rng.poisson(0.6, n)                                   # quejas del ultimo semestre
antiguedad_meses = rng.integers(2, 72, n)                                # meses como cliente

def z(v):                                    # pone cualquier columna en la misma escala
    v = np.asarray(v, dtype=float)
    return (v - v.mean()) / v.std()

riesgo = (1.0 * z(dias_sin_venir) - 0.9 * z(compras_mes) + 0.6 * z(quejas_6meses)
          - 0.4 * z(ticket_promedio) - 0.3 * z(antiguedad_meses))
se_va = (riesgo > np.quantile(riesgo, 0.65)).astype(int)   # el 35% de mayor riesgo se va
se_va = np.where(rng.random(n) < 0.08, 1 - se_va, se_va)   # 8% que rompe la regla: la vida real

clientes = pd.DataFrame({"compras_mes": compras_mes, "dias_sin_venir": dias_sin_venir,
                         "ticket_promedio": ticket_promedio, "quejas_6meses": quejas_6meses,
                         "antiguedad_meses": antiguedad_meses, "se_va": se_va})

print("Clientes en la base:", len(clientes))
print("Se fueron:", int(clientes["se_va"].sum()), "->", round(100 * clientes["se_va"].mean(), 1), "%")
clientes.head()

## 2️⃣ ¿Que son "capas y neuronas" y como se arman?

Aqui viene la unica parte tecnica de la Parte A, y se explica en dos minutos:

- Una **neurona** es una multiplicacion con una suma. Nada mas.
- Una **capa** es un grupo de neuronas que miran lo mismo al mismo tiempo. Nuestra red tiene tres:
  una de **16** neuronas que busca patrones, una de **8** que los resume, y una de **1** que da la respuesta final.
- La ultima neurona usa `sigmoid`, que obliga la respuesta a quedar entre 0 y 1. Eso se lee como
  **probabilidad de que el cliente se vaya**.

Antes de armarla, apartamos el 20% de los clientes para el examen final (la red no los va a ver mientras
estudia) y ponemos todas las columnas en la misma escala, porque 45.000 pesos y 3 quejas no se pueden sumar
tal cual.

**Que mirar en el resultado:** la tabla de `summary`, y en particular el total de `params`. Ese numero son
las perillas internas que la red va a ir ajustando sola. Con cinco columnas son unas 250.

In [ ]:
X = clientes.drop(columns=["se_va"]).to_numpy(dtype="float32")
y = clientes["se_va"].to_numpy(dtype="float32")

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

promedio = X_ent.mean(axis=0)          # las escalas se calculan SOLO con los datos de estudio
desviacion = X_ent.std(axis=0)
X_ent = (X_ent - promedio) / desviacion
X_pru = (X_pru - promedio) / desviacion

red = keras.Sequential([
    keras.Input(shape=(X.shape[1],)),                 # entran las 5 columnas del cliente
    keras.layers.Dense(16, activation="relu"),        # 16 neuronas que buscan patrones
    keras.layers.Dense(8,  activation="relu"),        # 8 que resumen esos patrones
    keras.layers.Dense(1,  activation="sigmoid"),     # 1 que responde: probabilidad de irse
])
red.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

print("Clientes para estudiar:", len(X_ent), "| clientes para el examen final:", len(X_pru))
red.summary()

## 3️⃣ ¿Como sabemos si la red aprendio o solo se memorizo la tabla?

La red estudia por **vueltas** (en el mundo tecnico se llaman *epocas*). En cada vuelta mira todos los
clientes, se equivoca, mide el error y se corrige. Aqui le damos hasta 60 vueltas.

Pero hay una trampa clasica y hay que verla con los ojos: la red puede **memorizar** en vez de aprender.
Por eso separamos un pedacito de los datos de estudio como *examen sorpresa* y dibujamos las dos lineas:

> **La regla para leer la grafica:** las dos lineas deben bajar juntas. Cuando la linea del examen sorpresa
> **se separa y empieza a subir** mientras la del entrenamiento sigue bajando, la red dejo de aprender y
> empezo a memorizar. Eso es el **sobreajuste**.

Para no dejarla llegar a ese punto le pusimos un freno automatico: si el examen sorpresa no mejora en 10
vueltas seguidas, se para y se queda con la mejor version que tuvo.

**Que mirar en el resultado:** las dos lineas de la grafica y si se separan o no. Y el numero de vueltas
efectivas: si es menor que 60, el freno actuo.

In [ ]:
freno = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

historia = red.fit(X_ent, y_ent, validation_split=0.2, epochs=60, batch_size=32,
                   verbose=0, callbacks=[freno])

plt.figure(figsize=(8, 4.5))
plt.plot(historia.history["loss"], label="Estudiando (entrenamiento)", linewidth=2)
plt.plot(historia.history["val_loss"], label="Examen sorpresa (validacion)", linewidth=2)
plt.title("Como fue aprendiendo la red vuelta por vuelta")
plt.xlabel("Vueltas de estudio")
plt.ylabel("Error (mas abajo es mejor)")
plt.legend()
plt.show()

print("Vueltas efectivas:", len(historia.history["loss"]), "de 60 permitidas")
print("Si son menos de 60, el freno automatico decidio que ya no valia la pena seguir.")

## 4️⃣ ¿Cuanto acierta con clientes que nunca vio?

Este es el numero honesto: el acierto sobre el 20% de clientes que apartamos al principio y que la red
nunca miro. Espere algo entre **85% y 92%**. No espere 100%: acuerdese del 8% de clientes que rompen la
regla a proposito. Un modelo que da 100% en la vida real casi siempre significa que hicimos algo mal.

Abajo tambien salen los cinco clientes con mas riesgo segun la red, con la verdad al lado. Esa es la
llamada que hace el equipo comercial el lunes.

**Que mirar en el resultado:** el porcentaje de acierto, y en la tablita si los de mayor riesgo si se fueron.

In [ ]:
perdida, acierto = red.evaluate(X_pru, y_pru, verbose=0)
print("Acierto con clientes que la red NUNCA vio:", round(acierto * 100, 1), "%")

probabilidad = red.predict(X_pru, verbose=0).ravel()
marcados = (probabilidad > 0.5).astype(int)
print("Clientes que la red marca como 'se va':", int(marcados.sum()), "de", len(marcados))
print("Clientes que se fueron de verdad:", int(y_pru.sum()))

revisar = pd.DataFrame({"riesgo_de_irse_pct": (probabilidad * 100).round(1),
                        "se_fue_de_verdad": y_pru.astype(int)})
revisar.sort_values("riesgo_de_irse_pct", ascending=False).head(5)

---

# 🅱️ PARTE B · El detector de dias raros

Hasta aqui todo era comodo: alguien nos habia dicho cual cliente se fue. **En el mundo real casi nunca
pasa eso.** Nadie le entrega una lista de "estos dias hubo fraude en la caja". Si la tuviera, no
necesitaria inteligencia artificial: ya sabria quien fue.

Entonces damos la vuelta al problema. En vez de ensenarle a la maquina como se ve un fraude, le ensenamos
como se ve **un dia normal**, hasta que se lo aprenda de memoria. Y despues le mostramos todos los dias del
ano y le preguntamos: *¿cual de estos no se te parece a lo que aprendiste?*

La herramienta se llama **autoencoder**. Funciona como un empleado al que le pedimos dos cosas seguidas:
**resuma** el informe del dia en cuatro frases, y ahora **vuelva a escribir el informe completo** solo con
esas cuatro frases. Si el dia era normal, lo reescribe casi igual. Si el dia era raro, la reescritura sale
torcida, y ese desajuste es la alarma.

## 5️⃣ ¿Y cuando nadie me dice cual dia fue fraude?

Armamos 1.000 dias de operacion de una tienda con seis columnas que salen de cualquier POS:
`ventas_dia`, `num_transacciones`, `ticket_promedio`, `descuentos_aplicados`, `devoluciones` y `horas_caja`.

De esos, el 95% son dias normales. El otro 5% esta **manipulado** con el patron clasico del fraude de caja:
descuentos y devoluciones inflados, con las ventas sospechosamente **planas** (siempre casi el mismo total,
porque el que manipula cuadra la caja a mano para que nadie note el hueco).

Y aqui va lo importante, que es a proposito: los fraudes van **graduados**, de finisimos a burdos. Los
burdos inflan el descuento hasta el 12% y dejan las ventas completamente planas. Los finos apenas suben el
descuento al 4,3% — un valor que cualquier dia bueno podria tener. Esos se van a escapar, y esta bien que
se escapen: asi es la vida real.

Guardamos la verdad en la columna `es_normal` **solo para calificar al detector al final**. El detector
nunca la ve.

**Que mirar en el resultado:** las dos ultimas lineas del texto. Fijese en que el rango de descuento de los
dias manipulados **se cruza** con el de los dias buenos. Por eso una regla simple de Excel no basta.

In [ ]:
rng = np.random.default_rng(2026)
n_malos = max(10, int(round(NUM_DIAS * PORCENTAJE_DIAS_MANIPULADOS / 100)))
n_buenos = NUM_DIAS - n_malos

# --- Dias normales: todo se mueve dentro de lo esperado ---
trans_b  = rng.normal(320, 40, n_buenos).clip(150, None).round(0)            # transacciones del dia
ticket_b = rng.normal(38000, 4500, n_buenos).clip(18000, None).round(0)      # pesos por transaccion
ventas_b = (trans_b * ticket_b * rng.normal(1, 0.02, n_buenos)).round(0)     # ventas = transacciones x ticket
desc_b   = (ventas_b * rng.normal(0.040, 0.010, n_buenos).clip(0.010, 0.090)).round(0)  # ~4% en descuentos
devo_b   = (ventas_b * rng.normal(0.015, 0.005, n_buenos).clip(0.002, 0.040)).round(0)  # ~1.5% devuelto
horas_b  = rng.normal(12.0, 0.7, n_buenos).clip(9, 15).round(1)              # horas de caja abierta

# --- Dias manipulados: graduados de finisimo (fuerza 0) a burdo (fuerza 1) ---
fuerza    = rng.uniform(0.0, 1.0, n_malos)
trans_m   = rng.normal(315, 40, n_malos).clip(150, None).round(0)
ticket_m  = rng.normal(37500, 4300, n_malos).clip(18000, None).round(0)
coherente = trans_m * ticket_m * rng.normal(1, 0.02, n_malos)                # lo que deberia haber vendido
plana     = np.median(ventas_b) * rng.normal(1.0, 0.012, n_malos)            # el total cuadrado a mano
ventas_m  = ((1 - fuerza) * coherente + fuerza * plana).round(0)             # mas burdo = mas plana
desc_m    = (ventas_m * (0.042 + fuerza * (0.120 - 0.042))).round(0)         # descuentos hasta el 12%
devo_m    = (ventas_m * (0.016 + fuerza * (0.055 - 0.016))).round(0)         # devoluciones hasta el 5.5%
horas_m   = rng.normal(12.0 + 1.4 * fuerza, 0.8).clip(9, 18).round(1)        # caja abierta hasta mas tarde

print("Dias normales:", n_buenos, "| dias manipulados:", n_malos)
print("Descuento sobre ventas -> dias buenos: hasta", round(100 * (desc_b / ventas_b).max(), 1),
      "%  |  dias manipulados: desde", round(100 * (desc_m / ventas_m).min(), 1), "%")
print("Se cruzan. Una regla de Excel del tipo 'descuento mayor al 7%' dejaria pasar los fraudes finos.")

Ahora juntamos los dos grupos en **una sola tabla revuelta**, igual que le llega a usted el
reporte del sistema: mil dias en fila, sin ninguna marca que diga cual es cual.

In [ ]:
dias = pd.DataFrame({
    "ventas_dia":           np.concatenate([ventas_b, ventas_m]),
    "num_transacciones":    np.concatenate([trans_b, trans_m]),
    "ticket_promedio":      np.concatenate([ticket_b, ticket_m]),
    "descuentos_aplicados": np.concatenate([desc_b, desc_m]),
    "devoluciones":         np.concatenate([devo_b, devo_m]),
    "horas_caja":           np.concatenate([horas_b, horas_m]),
    "es_normal":            np.concatenate([np.ones(n_buenos, int), np.zeros(n_malos, int)]),
})
dias = dias.sample(frac=1, random_state=42).reset_index(drop=True)   # se revuelven los dias
dias.insert(0, "dia", np.arange(1, len(dias) + 1))                  # numero de dia, sin pistas

COLUMNAS = ["ventas_dia", "num_transacciones", "ticket_promedio",
            "descuentos_aplicados", "devoluciones", "horas_caja"]

print("Tabla lista:", dias.shape[0], "dias y", len(COLUMNAS), "columnas de operacion.")
print("Promedio de cada columna segun la verdad (que el detector NO va a ver):")
dias.groupby("es_normal")[COLUMNAS].mean().round(0).rename(index={0: "manipulado", 1: "normal"})

## 6️⃣ ¿Por que entrenamos SOLO con los dias normales?

Esta es la decision que hace que todo funcione, y es la que mas se equivoca la gente.

Si le mostramos a la red los dias buenos **y** los manipulados, ella aprende que el fraude tambien es
normal, y despues no lo detecta. Es como pedirle a un cajero nuevo que aprenda el turno mirando a un
companero que roba: va a copiar el robo. Entonces la red **solo ve dias normales**. De esos, 80% para
estudiar y 20% para vigilar que no este memorizando.

Segundo detalle: ponemos las seis columnas en escala de 0 a 1 (`MinMax`), porque las ventas van en millones
y las horas de caja en decenas. Sin eso, las ventas se comerian a las demas columnas. Y los minimos y
maximos se calculan **solo con los dias de estudio**: si usaramos todos, le estariamos filtrando a la red
informacion de los dias que todavia no ha visto.

**Que mirar en el resultado:** los tres conteos. Confirme que los dias manipulados quedaron en cero durante
el entrenamiento.

In [ ]:
X_dias = dias[COLUMNAS].to_numpy(dtype="float32")
es_normal = dias["es_normal"].to_numpy() == 1

# La red solo ve dias normales. Los manipulados quedan afuera por completo.
X_norm_ent, X_norm_val = train_test_split(X_dias[es_normal], test_size=0.2, random_state=42)

minimo = X_norm_ent.min(axis=0)                                   # escalas: solo con lo que estudia
maximo = X_norm_ent.max(axis=0)
rango = np.where(maximo - minimo == 0, 1.0, maximo - minimo)       # evita dividir por cero

def escalar(M):
    return ((M - minimo) / rango).astype("float32")                # todo queda entre 0 y 1

X_ent_e   = escalar(X_norm_ent)
X_val_e   = escalar(X_norm_val)
X_todos_e = escalar(X_dias)                                        # los 1.000 dias, para el final

print("Dias normales para estudiar .....", len(X_ent_e))
print("Dias normales para vigilar ......", len(X_val_e))
print("Dias manipulados en el estudio ... 0  (de", int((~es_normal).sum()), "que hay en total)")

## 7️⃣ ¿Como se ve una red en forma de embudo?

Mire los numeros de las capas: **6 → 16 → 4 → 16 → 6**. Entra un dia con seis datos, se abre a 16 neuronas,
y despues se aprieta a **solo 4**. Esa es la parte que importa:

> **La esencia: resumir el informe de 8 paginas en 4 frases.** Y despues volver a escribir las 8 paginas
> completas usando unicamente esas 4 frases.

Si la red pudiera copiar los seis numeros tal cual, no aprenderia nada: seria una fotocopiadora. Al
obligarla a pasar por el cuello de cuatro neuronas, no le queda mas remedio que aprender **como se
relacionan** las columnas en un dia normal: que los descuentos son mas o menos el 4% de las ventas, que las
ventas son transacciones por ticket. Cuando llegue un dia con 25% de descuento, la reescritura le va a salir
mal, y ese error es la alarma.

La ultima capa usa `sigmoid` porque los datos estan entre 0 y 1, y tiene exactamente **seis** salidas: la
red tiene que devolver el dia completo, del mismo tamano que entro.

**Que mirar en el resultado:** en el `summary`, la columna de formas de salida bajando 16 → 4 → 16 → 6. Ese
es el embudo dibujado en numeros.

In [ ]:
detector = keras.Sequential([
    keras.Input(shape=(X_ent_e.shape[1],)),           # entra el dia completo: 6 numeros
    keras.layers.Dense(16, activation="relu"),        # se abre: mira el dato desde 16 angulos
    keras.layers.Dense(4,  activation="relu"),        # LA ESENCIA: el resumen en 4 frases
    keras.layers.Dense(16, activation="relu"),        # se vuelve a abrir
    keras.layers.Dense(X_ent_e.shape[1], activation="sigmoid"),  # reescribe el dia completo
])
detector.compile(optimizer="adam", loss="mae")        # mae = de cuanto se equivoca al reescribir

detector.summary()

## 8️⃣ ¿Como se entrena una red que se copia a si misma?

Aqui esta el truco mas raro y mas bonito de todo el notebook. Fijese en la linea del entrenamiento:

```
detector.fit(X_ent_e, X_ent_e, ...)
```

**La pregunta y la respuesta son el mismo dato.** Le decimos: "aqui esta el dia, y la respuesta correcta es
ese mismo dia". La red se **copia a si misma**. Como la obligamos a pasar por el cuello de cuatro neuronas,
copiar bien solo es posible si de verdad entendio como funciona un dia normal.

Otra vez ponemos el freno automatico: hasta las vueltas que diga el panel de control, y si en 10 vueltas
seguidas no mejora, se para y se queda con su mejor version.

**Que mirar en el resultado:** las dos lineas deben bajar y quedarse abajo, juntas. Ese "quedarse abajo"
significa que la red ya sabe reescribir un dia normal con muy poco error.

In [ ]:
freno = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

historia_det = detector.fit(X_ent_e, X_ent_e,                       # la red se copia a si misma
                            validation_data=(X_val_e, X_val_e),
                            epochs=EPOCAS_MAXIMAS, batch_size=32,
                            shuffle=True, verbose=0, callbacks=[freno])

plt.figure(figsize=(8, 4.5))
plt.plot(historia_det.history["loss"], label="Estudiando (dias normales)", linewidth=2)
plt.plot(historia_det.history["val_loss"], label="Vigilancia (dias normales apartados)", linewidth=2)
plt.title("Que tanto le cuesta a la red reescribir un dia normal")
plt.xlabel("Vueltas de estudio")
plt.ylabel("Error de reconstruccion")
plt.legend()
plt.show()

print("Vueltas efectivas:", len(historia_det.history["loss"]), "de", EPOCAS_MAXIMAS, "permitidas")

## 9️⃣ ¿Donde ponemos la raya entre "normal" y "raro"?

Ya tenemos una red que sabe reescribir dias normales. Ahora medimos, para cada dia normal, **cuanto se
equivoco** al reescribirlo. Eso es el **error de reconstruccion**: un solo numero por dia.

La mayoria de los dias normales va a tener un error chiquito. Con eso armamos la raya:

> **umbral = promedio del error + 2 desviaciones**

En castellano: "lo normal es equivocarse esto; si se equivoca bastante mas que lo normal, es porque el dia
no se le parece a nada de lo que aprendio". Ese 2 es la perilla `SENSIBILIDAD_DEL_DETECTOR` del panel: con
1.0 el detector se vuelve desconfiado y marca muchos dias buenos; con 3.0 solo marca lo escandaloso.

**Que mirar en el resultado:** el histograma debe ser una montana pegada a la izquierda, y la **linea roja**
debe quedar a la derecha de casi toda la montana. Si la linea le corta la montana por la mitad, subale la
sensibilidad.

In [ ]:
def error_por_dia(modelo, M):
    reescrito = modelo.predict(M, verbose=0)          # la red reescribe cada dia
    return np.mean(np.abs(reescrito - M), axis=1)     # de cuanto se equivoco, dia por dia

error_normales = error_por_dia(detector, X_ent_e)
umbral = error_normales.mean() + SENSIBILIDAD_DEL_DETECTOR * error_normales.std()

plt.figure(figsize=(8.5, 4.5))
plt.hist(error_normales, bins=40, color="#29AC8A", edgecolor="white")
plt.axvline(umbral, color="#D54E48", linewidth=2.5, label="Umbral de sospecha")
plt.title("Que tan bien reescribe la red los dias que si son normales")
plt.xlabel("Error de reconstruccion")
plt.ylabel("Numero de dias")
plt.legend()
plt.show()

print("Error promedio en dias normales:", round(float(error_normales.mean()), 5))
print("Umbral de sospecha (promedio +", SENSIBILIDAD_DEL_DETECTOR, "desviaciones):", round(float(umbral), 5))

## 🔟 ¿Que dias no se parecen a un dia normal?

Momento de la verdad. Le pasamos a la red **los 1.000 dias**, incluidos los manipulados que nunca vio, y
medimos el error de cada uno. Todo dia que pase la raya queda marcado.

En la grafica pintamos las dos poblaciones con colores distintos usando la columna `es_normal`. Ojo: eso lo
hacemos **para calificar al detector**, no para que el decida. El detector solo miro el error.

**Que mirar en el resultado:** dos montanas separadas. La verde (dias normales) pegada a la izquierda, y la
roja (dias manipulados) corrida a la derecha, del otro lado de la linea. Donde las dos montanas se tocan,
ahi estan los fraudes finos que se van a escapar.

In [ ]:
dias["error_reconstruccion"] = error_por_dia(detector, X_todos_e)
dias["marcado_como_raro"] = (dias["error_reconstruccion"] > umbral).astype(int)

plt.figure(figsize=(9.5, 4.5))
plt.hist(dias.loc[dias["es_normal"] == 1, "error_reconstruccion"], bins=45,
         color="#29AC8A", alpha=0.80, label="Dias normales", edgecolor="white")
plt.hist(dias.loc[dias["es_normal"] == 0, "error_reconstruccion"], bins=45,
         color="#D54E48", alpha=0.85, label="Dias manipulados", edgecolor="white")
plt.axvline(umbral, color="black", linestyle="--", linewidth=2, label="Umbral de sospecha")
plt.title("Los dias que la red no logra reescribir son los dias raros")
plt.xlabel("Error de reconstruccion")
plt.ylabel("Numero de dias")
plt.legend()
plt.show()

print("Dias marcados para revisar:", int(dias["marcado_como_raro"].sum()), "de", len(dias))

## 1️⃣1️⃣ ¿Cual es la lista que me llevo a la reunion?

Esta es la celda que justifica todo el notebook. Ni la red, ni el umbral, ni el histograma sirven de nada si
al final usted no puede decirle a alguien **cuales dias revisar el lunes a primera hora**.

Ordenamos los 1.000 dias por error, de peor a mejor, y nos quedamos con los diez de arriba. Al lado ponemos
el veredicto real, que en su empresa no lo va a tener: eso es solo para que hoy midamos si el detector sirve.

**Que mirar en el resultado:** la frase de arriba. De los 10 dias mas sospechosos, cuantos estaban
manipulados de verdad. Si son 8, 9 o 10, la lista es oro: el auditor revisa diez dias en vez de mil.

In [ ]:
sospechosos = dias.sort_values("error_reconstruccion", ascending=False).head(10).copy()
sospechosos["veredicto_real"] = np.where(sospechosos["es_normal"] == 1, "normal", "MANIPULADO")
sospechosos["pct_descuento"] = (100 * sospechosos["descuentos_aplicados"] / sospechosos["ventas_dia"]).round(1)

encontrados = int((sospechosos["es_normal"] == 0).sum())
print("De los 10 dias mas sospechosos,", encontrados, "estaban manipulados de verdad.")
print("Revisar 10 dias en vez de", len(dias), "es lo que esta comprando con esto.")

sospechosos[["dia", "ventas_dia", "descuentos_aplicados", "devoluciones",
             "pct_descuento", "error_reconstruccion", "veredicto_real"]].round(4)

## 1️⃣2️⃣ ¿En que se equivoca el detector y cuanto cuesta cada error?

Ningun detector es perfecto, y el gerente que lo compra tiene que saber **de que manera** falla. La matriz
de abajo tiene cuatro casillas y cada una cuesta plata distinta:

| Casilla | Que paso | Cuanto cuesta |
|---|---|---|
| Arriba a la izquierda | Dia normal, no lo marco | Nada. Asi debe ser. |
| Abajo a la derecha | Dia manipulado, lo marco | **Aqui esta el valor.** Fraude encontrado. |
| Arriba a la derecha | Dia normal, lo marco igual | **Falsa alarma.** Cuesta horas de auditoria y desgasta la confianza del equipo. |
| Abajo a la izquierda | Dia manipulado, se le paso | **El error caro.** El hueco sigue abierto. |

La perilla `SENSIBILIDAD_DEL_DETECTOR` mueve el balance entre las dos ultimas: bajarla encuentra mas fraude
pero llena de falsas alarmas; subirla limpia las falsas alarmas pero deja pasar los fraudes finos. **Esa
decision no es tecnica, es gerencial**, y es suya.

**Que mirar en el resultado:** los tres numeros de abajo. Espere encontrar entre el **75% y el 95%** de los
fraudes, con unas cuantas decenas de falsas alarmas. Si le sale 100%, desconfie: los datos estarian
demasiado limpios para ser reales.

In [ ]:
era_anomalia = (dias["es_normal"] == 0).astype(int)      # 1 = manipulado de verdad
lo_marco     = dias["marcado_como_raro"].astype(int)     # 1 = el detector lo marco
matriz = confusion_matrix(era_anomalia, lo_marco, labels=[0, 1])

plt.figure(figsize=(6, 4.8))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, annot_kws={"size": 14},
            xticklabels=["Normal", "Anomalia"], yticklabels=["Normal", "Anomalia"])
plt.title("Lo que era en realidad, contra lo que dijo el detector")
plt.xlabel("Lo que dijo el detector")
plt.ylabel("Lo que era en realidad")
plt.show()

vn, fp, fn, vp = matriz.ravel()
print("Fraudes encontrados ......", vp, "de", vp + fn, "->", round(100 * vp / max(vp + fn, 1), 1), "%")
print("Fraudes que se escaparon .", fn, "(los mas finos, los que casi no inflaron nada)")
print("Falsas alarmas ...........", fp, "dias buenos que igual toca revisar")

---

## 🏆 La frase que se lleva de esta clase

> ### "No le ensenamos a la maquina como se ve un fraude — le ensenamos como se ve un dia normal, y ella nos dice que dias no se parecen."

Eso es lo que cambia el juego. Para entrenar un detector de fraude a la manera vieja usted necesitaba una
lista de fraudes confirmados, y esa lista casi nunca existe. Con un autoencoder solo necesita lo que si
tiene de sobra: **meses y meses de operacion normal**.

Y sirve para todo lo que se pueda medir en filas y columnas: facturas de proveedores con montos y plazos
fuera de rango, consumo de insumos por servicio en un hospital, sobrecostos por capitulo de obra, contratos
con valores raros frente a su categoria, transacciones de un mismo usuario a horas imposibles.

---

## 🏁 Ahora con SUS datos

Todo lo que vio arriba corrio con datos inventados en la primera celda. Para volverlo suyo hay que cambiar
**una sola cosa**: de donde salen las tablas `clientes` y `dias`.

**Paso 1 — arme su archivo.** En Excel deje una hoja donde cada **fila** es un dia (o una factura, o un
contrato, o un servicio) y cada **columna** un numero de ese dia. Sin celdas combinadas, sin totales al pie,
sin filas de encabezado dobles. Guardelo como **CSV (delimitado por comas)**.

**Paso 2 — subalo a Colab.** En el panel de la izquierda, el icono de la carpeta 📁, y arrastre el CSV ahi.
O agregue una celda nueva de codigo con estas dos lineas, corrala y elija el archivo:

- `from google.colab import files`
- `subido = files.upload()`

**Paso 3 — cambie la lectura.** Reemplace la celda que inventa los dias por:

- `dias = pd.read_csv("mi_archivo.csv")`
- y en la lista `COLUMNAS` ponga los nombres exactos de **sus** columnas numericas.

**Paso 4 — cuidado con dos cosas.** Primera: si usted **no tiene** la columna `es_normal`, no puede sacar la
matriz de confusion ni el histograma de dos colores; solo la lista de los diez mas sospechosos, que es
precisamente lo que necesita. Segunda: para entrenar solo con lo normal, use un periodo que usted sepa que
estuvo limpio (por ejemplo un trimestre ya auditado).

**Paso 5 — no arranque con seis columnas raras.** Empiece con cuatro o cinco columnas que usted entienda
bien. Un detector que usted puede explicar en la junta vale mas que uno que nadie sabe leer.

### 🤖 Preguntele a su Claude

Abra Claude, adjunte este notebook y pidale con sus propias palabras. Tres arranques que sirven:

> **Para adaptarlo:** "Adapta este notebook a mis datos. Mi tabla se llama `movimientos.csv` y tiene estas
> columnas: [pegue aqui los nombres y diga que significa cada una]. Quiero detectar [lo que quiere detectar].
> Cambiame la celda de generacion de datos por la lectura de mi CSV y ajusta la lista COLUMNAS. Explicame en
> espanol sencillo que cambiaste y por que."

> **Para entender un resultado:** "En la matriz de confusion me salieron [pegue los cuatro numeros]. En mi
> caso una falsa alarma cuesta como dos horas de un auditor y un fraude que se escapa cuesta unos 3 millones
> de pesos. ¿En que valor deberia dejar SENSIBILIDAD_DEL_DETECTOR y por que?"

> **Para llevarlo a la junta:** "Con los resultados de este notebook armame una diapositiva de una pagina
> para mi comite: que problema resuelve, que encontro, cuanto tiempo de auditoria ahorra y que necesito para
> ponerlo en produccion. Sin palabras tecnicas."

---

**Diplomado en IA · UNAC · Clase 04 · Si algo falla, copie el error completo y pegueselo a su Claude: el sabe de este notebook.**